In [3]:
from pymavlink import mavutil


# =========================
# Change this path only
# =========================
TLOG_PATH = "has_ROI.tlog"



def cmd_name(cmd_id):
    try:
        e = mavutil.mavlink.enums["MAV_CMD"].get(int(cmd_id))
        return e.name if e else f"UNKNOWN_MAV_CMD_{cmd_id}"
    except Exception:
        return f"UNKNOWN_MAV_CMD_{cmd_id}"


def frame_name(frame_id):
    try:
        e = mavutil.mavlink.enums["MAV_FRAME"].get(int(frame_id))
        return e.name if e else f"UNKNOWN_MAV_FRAME_{frame_id}"
    except Exception:
        return f"UNKNOWN_MAV_FRAME_{frame_id}"


def print_mission_item(msg):
    msg_type = msg.get_type()

    seq = int(getattr(msg, "seq", -1))
    frame = int(getattr(msg, "frame", -1))
    command = int(getattr(msg, "command", -1))

    p1 = float(getattr(msg, "param1", 0.0))
    p2 = float(getattr(msg, "param2", 0.0))
    p3 = float(getattr(msg, "param3", 0.0))
    p4 = float(getattr(msg, "param4", 0.0))

    x = getattr(msg, "x", None)
    y = getattr(msg, "y", None)
    z = getattr(msg, "z", None)

    if msg_type == "MISSION_ITEM_INT":
        lat = float(x) / 1e7 if x is not None else None
        lon = float(y) / 1e7 if y is not None else None
        alt = float(z) if z is not None else None
    else:
        lat = float(x) if x is not None else None
        lon = float(y) if y is not None else None
        alt = float(z) if z is not None else None

    print("=" * 70)
    print(f"Message Type : {msg_type}")
    print(f"Seq          : {seq}")
    print(f"Command      : {command} ({cmd_name(command)})")
    print(f"Frame        : {frame} ({frame_name(frame)})")
    print(f"Current      : {getattr(msg, 'current', None)}")
    print(f"Autocontinue : {getattr(msg, 'autocontinue', None)}")
    print(f"Param1       : {p1}")
    print(f"Param2       : {p2}")
    print(f"Param3       : {p3}")
    print(f"Param4       : {p4}")
    print(f"Latitude     : {lat}")
    print(f"Longitude    : {lon}")
    print(f"Altitude     : {alt}")


def extract_and_print_mission_items(tlog_path):
    mav = mavutil.mavlink_connection(tlog_path, robust_parsing=True)

    mission_count = None
    item_count = 0

    while True:
        msg = mav.recv_match(blocking=False)

        if msg is None:
            break

        msg_type = msg.get_type()

        if msg_type == "MISSION_COUNT":
            mission_count = int(getattr(msg, "count", -1))
            print("\n" + "#" * 70)
            print(f"MISSION_COUNT found: {mission_count}")
            print("#" * 70)

        elif msg_type in ("MISSION_ITEM", "MISSION_ITEM_INT"):
            item_count += 1
            print_mission_item(msg)

    print("\n" + "#" * 70)
    print(f"Done. Total mission items found: {item_count}")

    if mission_count is not None:
        print(f"Last reported mission count: {mission_count}")

    print("#" * 70)

extract_and_print_mission_items(TLOG_PATH)


######################################################################
MISSION_COUNT found: 0
######################################################################

######################################################################
MISSION_COUNT found: 0
######################################################################

######################################################################
MISSION_COUNT found: 0
######################################################################

######################################################################
MISSION_COUNT found: 19
######################################################################
Message Type : MISSION_ITEM_INT
Seq          : 0
Command      : 530 (MAV_CMD_SET_CAMERA_MODE)
Frame        : 2 (MAV_FRAME_MISSION)
Current      : 1
Autocontinue : 1
Param1       : 0.0
Param2       : 2.0
Param3       : nan
Param4       : nan
Latitude     : -214.7483648
Longitude    : -214.7483648
Altitude     : nan
Message Type : MISS

In [3]:
import json
from pprint import pprint

try:
    from pymavlink.dialects.v20 import common as mavlink2
    MAV_CMD_ENUM = mavlink2.enums.get("MAV_CMD", {})
    MAV_FRAME_ENUM = mavlink2.enums.get("MAV_FRAME", {})
except Exception:
    MAV_CMD_ENUM = {}
    MAV_FRAME_ENUM = {}


def mav_cmd_name(command):
    try:
        command = int(command)
        return MAV_CMD_ENUM[command].name if command in MAV_CMD_ENUM else f"UNKNOWN_MAV_CMD_{command}"
    except Exception:
        return "UNKNOWN_MAV_CMD"


def mav_frame_name(frame):
    try:
        frame = int(frame)
        return MAV_FRAME_ENUM[frame].name if frame in MAV_FRAME_ENUM else f"UNKNOWN_FRAME_{frame}"
    except Exception:
        return "UNKNOWN_FRAME"


def normalize_params(params):
    params = list(params or [])
    while len(params) < 7:
        params.append(None)
    return params[:7]


def extract_mission_items_from_plan(plan_file):
    with open(plan_file, "r") as f:
        plan = json.load(f)

    mission = plan.get("mission", {})
    items = mission.get("items", [])

    extracted = []

    def walk(items, parent_type=None):
        for item in items:
            item_type = item.get("type", "Unknown")

            if item_type == "SimpleItem":
                command = item.get("command")
                frame = item.get("frame")
                params = normalize_params(item.get("params"))

                p1, p2, p3, p4, lat, lon, alt = params

                extracted.append({
                    "item_type": "SimpleItem",
                    "doJumpId": item.get("doJumpId"),
                    "command": command,
                    "command_name": mav_cmd_name(command),
                    "frame": frame,
                    "frame_name": mav_frame_name(frame),
                    "autoContinue": item.get("autoContinue"),
                    "param1": p1,
                    "param2": p2,
                    "param3": p3,
                    "param4": p4,
                    "latitude": lat,
                    "longitude": lon,
                    "altitude": alt,
                    "parent_type": parent_type,
                    "raw": item,
                })

            elif item_type == "ComplexItem":
                complex_type = item.get("complexItemType", "UnknownComplexItem")

                extracted.append({
                    "item_type": "ComplexItem",
                    "complexItemType": complex_type,
                    "parent_type": parent_type,
                    "raw": item,
                })

                nested_items = item.get("Items") or item.get("items") or []
                if nested_items:
                    walk(nested_items, parent_type=complex_type)

            elif item_type == "MissionSettings":
                extracted.append({
                    "item_type": "MissionSettings",
                    "parent_type": parent_type,
                    "raw": item,
                })

            else:
                extracted.append({
                    "item_type": item_type,
                    "parent_type": parent_type,
                    "raw": item,
                })

    walk(items)
    return extracted


def print_mission_items(mission_items):
    print("\n========== EXTRACTED MISSION ITEMS ==========\n")

    for i, item in enumerate(mission_items, start=1):
        print(f"Item {i}")
        print(f"  item_type     : {item.get('item_type')}")

        if item.get("item_type") == "SimpleItem":
            print(f"  doJumpId      : {item.get('doJumpId')}")
            print(f"  command       : {item.get('command')} ({item.get('command_name')})")
            print(f"  frame         : {item.get('frame')} ({item.get('frame_name')})")
            print(f"  autoContinue  : {item.get('autoContinue')}")
            print(f"  param1        : {item.get('param1')}")
            print(f"  param2        : {item.get('param2')}")
            print(f"  param3        : {item.get('param3')}")
            print(f"  param4        : {item.get('param4')}")
            print(f"  latitude      : {item.get('latitude')}")
            print(f"  longitude     : {item.get('longitude')}")
            print(f"  altitude      : {item.get('altitude')}")

        elif item.get("item_type") == "ComplexItem":
            print(f"  complexType   : {item.get('complexItemType')}")

        elif item.get("item_type") == "MissionSettings":
            print("  Mission settings item found.")

        else:
            print("  Unknown item type found.")

        if item.get("parent_type"):
            print(f"  parent_type   : {item.get('parent_type')}")

        print("-" * 60)


plan_file = "mission_alt_yaw_speed.plan"   # change path if needed

mission_items = extract_mission_items_from_plan(plan_file)
print_mission_items(mission_items)


========== EXTRACTED MISSION ITEMS ==========

Item 1
  item_type     : SimpleItem
  doJumpId      : 1
  command       : 22 (MAV_CMD_NAV_TAKEOFF)
  frame         : 3 (MAV_FRAME_GLOBAL_RELATIVE_ALT)
  autoContinue  : True
  param1        : 0
  param2        : 0
  param3        : 0
  param4        : None
  latitude      : 47.3979713
  longitude     : 8.5461636
  altitude      : 49.9872
------------------------------------------------------------
Item 2
  item_type     : SimpleItem
  doJumpId      : 2
  command       : 16 (MAV_CMD_NAV_WAYPOINT)
  frame         : 3 (MAV_FRAME_GLOBAL_RELATIVE_ALT)
  autoContinue  : True
  param1        : 0
  param2        : 0
  param3        : 0
  param4        : None
  latitude      : 47.39812292
  longitude     : 8.54368057
  altitude      : 49.9872
------------------------------------------------------------
Item 3
  item_type     : SimpleItem
  doJumpId      : 3
  command       : 16 (MAV_CMD_NAV_WAYPOINT)
  frame         : 3 (MAV_FRAME_GLOBAL_RELATIVE_